In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M15, 0, 200)
#     rates_frame = pd.DataFrame(rates)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 21)
#     rates_frame['rsi'] = get_rsi(rates_frame['close'], r)
    rates_frame['sma'] = rates_frame['close'].rolling(window=50).mean()

    rates_frame['sma1'] = rates_frame['close'].rolling(window=9).mean()
    rates_frame['sma2'] = rates_frame['close'].rolling(window=21).mean()
    rates_frame = rates_frame[rates_frame['sma1'].notna()]
    rates_frame = rates_frame[rates_frame['sma2'].notna()]

    # Calculate Supertren
    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)


def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0
    old = 0
    old_pp = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            try:
                print(a.iloc[-2].name)
                old = a.iloc[-2].close
                pp = mt5.positions_get(ticket=result_buy.order)[0].profit
                old_pp= pp
                
            except:
                pass

            if a.iloc[-3].rsi<30 and a.iloc[-2].rsi< 29.0 and a.iloc[-1].rsi >=34.0:
                result_buy = Action(symbol, lot, buy)
                print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                buy_check = 1
                old = a.iloc[-2].close

                sell_check = 0

        if sell_check==1:
            pp = mt5.positions_get(ticket=result_sell.order)[0].profit
            if pp<= -10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
            if pp >= 10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close


                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0

        if buy_check==1:
            if c == 1:
                result_buy_1 = Action_close(result_buy_1.order, symbol, buy, lot)     #Action_close

                if result_buy_1.comment == "Requote":
                    result_buy_1 = Action_close(result_buy_1.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy_1.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy_1.comment}")
                c = 0
            pp = mt5.positions_get(ticket=result_buy.order)[0].profit
            if a.iloc[-1].rsi<a.iloc[-2].rsi:
                result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                if result_buy.comment == "Requote":
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                buy_check = 0
            elif c==0:
                c =1
                result_buy_1 = Action(symbol, lot, buy)
                print(f"result_buy_1  Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy_1.order} ||| result_comment-->{result_buy_1.comment}")
                
        time.sleep(1)

for symbol in ['BTCUSD']:
    run(symbol)
    

BTCUSD
2024-08-06 17:30:00
2024-08-06 17:45:00
2024-08-06 18:00:00
2024-08-06 18:15:00
2024-08-06 18:30:00
2024-08-06 18:45:00
2024-08-06 19:00:00
2024-08-06 19:15:00
2024-08-06 19:30:00
2024-08-06 19:45:00
2024-08-06 20:00:00
2024-08-06 20:15:00
2024-08-06 20:30:00
2024-08-06 20:45:00
2024-08-06 21:00:00
2024-08-06 21:15:00
2024-08-06 21:30:00
2024-08-06 21:45:00
2024-08-06 22:00:00
2024-08-06 22:15:00
2024-08-06 22:30:00
2024-08-06 22:45:00
2024-08-06 23:00:00
2024-08-06 23:15:00
2024-08-06 23:30:00
2024-08-06 23:45:00
2024-08-07 00:00:00
2024-08-07 00:15:00
2024-08-07 00:30:00
2024-08-07 00:45:00
2024-08-07 01:00:00
2024-08-07 01:15:00
2024-08-07 01:30:00
2024-08-07 01:45:00
2024-08-07 02:00:00
2024-08-07 02:15:00
2024-08-07 02:30:00
2024-08-07 02:45:00
2024-08-07 03:00:00
2024-08-07 03:15:00
2024-08-07 03:30:00
2024-08-07 03:45:00
2024-08-07 04:00:00
2024-08-07 04:15:00
2024-08-07 04:30:00
2024-08-07 04:45:00
2024-08-07 05:00:00
2024-08-07 05:15:00
2024-08-07 05:30:00
2024-08-07 05